In [1]:
# PHASE 2 - NOTEBOOK D2: downstream evaluation of ALL 18 checkpoints (3 backbones x neighbor/self x seeds 42,43,44) + 3-seed summary
# Kaggle setup: GPU on, Internet on.  Add your checkpoints dataset (peesarisathvikreddy/adjclv2-checkpoints) as input,
# after uploading the new files  adjclv2_<backbone>_<mode>_s43/s44_best.pt  as a NEW VERSION of it.
# All seeds in Config.SEEDS are re-extracted; only the 3 ImageNet controls are reused from your first D run (hard-coded below).
# Progress is saved after every architecture, so a crash costs at most one architecture.
# Re-running resumes: features (.npy) and per-model results (.json) are cached in /kaggle/working.
!pip install -q timm datasets kagglehub

"""
====================================================================
PART-II / PHASE 2 -- DOWNSTREAM EVALUATION of the adjacency-contrastive checkpoints
Frozen linear probe on Camelyon17-WILDS: primary split (train centers 0,1,3,4 -> test
center 2, patient-bootstrap CI) + leave-one-center-out (LOCO) + a patch-level center-ID
probe.  Same probe recipe as Part I (StandardScaler + LogisticRegression C=1.0,
balanced class weights, max_iter 2000; 1000 patient-level bootstrap resamples, seed 7).

All models are pushed through ONE pass over the 455,954 patches (each image is decoded once).
Controls evaluated in the same pass:
  * ImageNet ResNet18 @96px  -> must reproduce the Part-I ImageNet baseline (LOCO ~0.9528)
  * ImageNet ViT-Small / Swin-Tiny @224px -> same-architecture, same-init controls
    (this is the baseline the contrastive pretraining has to beat for the Transformers)
Part-I numbers (Table 1 of the paper) are printed alongside for reference.
====================================================================
"""
import os, gc, json, glob, random, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
from tqdm import tqdm

warnings.filterwarnings('ignore')


class Config:
    HF_DATASET = "wltjr1007/Camelyon17-WILDS"
    # where to look for the adjclv2_*_best.pt files (searched recursively)
    CKPT_SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working/checkpoints"]
    KAGGLE_CKPT_DATASET = "peesarisathvikreddy/adjclv2-checkpoints"   # fallback if not attached
    SEEDS = [42, 43, 44]    # all seeds to evaluate (missing checkpoints are skipped)
    CKPT_TAG = "adjclv2"   # v2 = same-slide hard negatives (v1 files were "adjcl")
    FEATURE_DIR = "/kaggle/working/p2_features"
    RESULTS_DIR = "/kaggle/working/p2_eval"

    EXTRACT_BATCH = 128
    NUM_WORKERS = 4
    USE_AMP_EXTRACT = False        # keep False to match the fp32 Part-I features

    ALL_CENTERS = [0, 1, 2, 3, 4]
    TEST_CENTER = 2
    LR_C, LR_MAX_ITER, PROBE_SEED = 1.0, 2000, 42
    BOOT_N, BOOT_SEED = 1000, 7
    LOCO_FOLDS = [0, 1, 2, 3, 4]

    # patch-level center-ID probe
    CID_TRAIN_PER_CENTER, CID_TEST_PER_CENTER, CID_VAL_FRAC = 3000, 1000, 0.25

    # model registry: name -> spec  (size = input px; kind adjcl|imagenet)
    MODELS = {}
    for _bb, _sz in (('resnet18', 96), ('vit_small', 224), ('swin_tiny', 224)):
        for _seed in SEEDS:
            for _mode in ('neighbor', 'self'):
                MODELS[f'adjcl_{_bb}_{_mode}_s{_seed}'] = dict(kind='adjcl', backbone=_bb, mode=_mode, size=_sz, seed=_seed)

    # Results of the first D run (seed 42 + ImageNet controls): primary AUROC, LOCO mean, center-ID %
    SEED42 = {
        ('resnet18', 'neighbor'): (0.9744, 0.9780, 44.2), ('resnet18', 'self'): (0.9591, 0.9651, 42.6),
        ('vit_small', 'neighbor'): (0.9849, 0.9844, 78.1), ('vit_small', 'self'): (0.9737, 0.9733, 70.5),
        ('swin_tiny', 'neighbor'): (0.9879, 0.9834, 81.3), ('swin_tiny', 'self'): (0.9771, 0.9748, 80.3)}
    IMAGENET = {'resnet18': (0.9473, 0.9528, 78.4), 'vit_small': (0.9726, 0.9573, 89.1), 'swin_tiny': (0.9735, 0.9687, 88.8)}

    # Part-I reference (paper Table 1: primary AUROC, LOCO mean)
    PART1_REF = {
        'Recon ResNet18 (Part I)': (0.8848, 0.9148), 'Recon ViT-Small (Part I)': (0.9605, 0.9656),
        'Recon Swin-Tiny (Part I)': (0.9615, 0.9615), 'ImageNet ResNet18 (Part I)': (0.9473, 0.9528),
        'Stain Augmentation': (0.9816, 0.9884), 'Kaiko ViT-S/16': (0.9825, 0.9873),
        'Phikon': (0.9508, 0.9818), 'Lunit ViT-S/16': (0.9783, 0.9814), 'Random init': (0.6393, 0.7797)}


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


# ---------------------------------------------------------------- encoders
class Encoder(nn.Module):
    """Backbone built exactly like AdjCLModel / the Part-I ReconstructionModel (keys `backbone.*`)."""
    def __init__(self, backbone, pretrained):
        super().__init__()
        self.pool = None
        if backbone == 'resnet18':
            w = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
            net = models.resnet18(weights=w)
            self.backbone = nn.Sequential(*list(net.children())[:-1])
            self.pool = nn.AdaptiveAvgPool2d(1); self.dim = 512
        elif backbone == 'vit_small':
            import timm
            self.backbone = timm.create_model('vit_small_patch16_224', pretrained=pretrained, num_classes=0, img_size=224)
            self.dim = 384
        elif backbone == 'swin_tiny':
            import timm
            self.backbone = timm.create_model('swin_tiny_patch4_window7_224', pretrained=pretrained, num_classes=0)
            self.dim = 768
        else:
            raise ValueError(backbone)

    @torch.no_grad()
    def encode(self, x):
        f = self.backbone(x)
        if f.dim() == 4:
            f = torch.flatten(self.pool(f), 1)
        elif f.dim() == 3:
            f = f.mean(1)
        return f


def ensure_checkpoint_roots():
    """If the checkpoints dataset is not attached, fetch it with kagglehub and add it to the search roots."""
    probe = f"{Config.CKPT_TAG}_resnet18_neighbor_s{Config.SEEDS[0]}_best.pt"
    for root in Config.CKPT_SEARCH_ROOTS:
        if glob.glob(os.path.join(root, '**', probe), recursive=True):
            return
    try:
        import kagglehub
        path = kagglehub.dataset_download(Config.KAGGLE_CKPT_DATASET)
        print("✓ checkpoints downloaded via kagglehub:", path)
        Config.CKPT_SEARCH_ROOTS.append(path)
    except Exception as e:
        print(f"⚠ could not download {Config.KAGGLE_CKPT_DATASET}: {e}")


def find_checkpoint(backbone, mode, seed):
    fname = f"{Config.CKPT_TAG}_{backbone}_{mode}_s{seed}_best.pt"
    for root in Config.CKPT_SEARCH_ROOTS:
        hits = glob.glob(os.path.join(root, '**', fname), recursive=True)
        if hits:
            return hits[0]
    return None


def build_model(name):
    spec = Config.MODELS[name]
    if spec['kind'] == 'imagenet':
        return Encoder(spec['backbone'], pretrained=True)
    path = find_checkpoint(spec['backbone'], spec['mode'], spec['seed'])
    if path is None:
        return None
    enc = Encoder(spec['backbone'], pretrained=False)
    ck = torch.load(path, map_location='cpu')
    missing, unexpected = enc.load_state_dict(ck['model_state_dict'], strict=False)
    bad_unexpected = [k for k in unexpected if not k.startswith('head.')]
    assert not missing and not bad_unexpected, f"{name}: missing={missing[:3]} unexpected={bad_unexpected[:3]}"
    print(f"  [{name}] loaded {path}  (epoch {ck.get('epoch','?')}, val loss {ck.get('best_val', float('nan')):.4f})")
    return enc


# ---------------------------------------------------------------- one-pass feature extraction
class MultiSizeDataset(Dataset):
    def __init__(self, hf, sizes):
        self.hf, self.sizes = hf, sorted(sizes)
        norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        self.tf = {s: transforms.Compose([transforms.Resize((s, s)), transforms.ToTensor(), norm]) for s in self.sizes}

    def __len__(self):
        return len(self.hf)

    def __getitem__(self, i):
        img = self.hf[int(i)]['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return {s: self.tf[s](img) for s in self.sizes}


def feat_path(name):
    return f"{Config.FEATURE_DIR}/{name}.npy"


def extract_missing(hf, names, device):
    todo = [n for n in names if not os.path.exists(feat_path(n))]
    if not todo:
        return
    mods = {}
    for n in todo:
        m = build_model(n)
        if m is None:
            print(f"  [{n}] checkpoint not found -> skipped")
            continue
        mods[n] = m.to(device).eval()
    if not mods:
        return
    sizes = {Config.MODELS[n]['size'] for n in mods}
    N = len(hf)
    out = {n: np.empty((N, mods[n].dim), dtype=np.float32) for n in mods}
    loader = DataLoader(MultiSizeDataset(hf, sizes), batch_size=Config.EXTRACT_BATCH, shuffle=False,
                        num_workers=Config.NUM_WORKERS, pin_memory=device.type == 'cuda')
    pos, t0 = 0, time.time()
    amp = Config.USE_AMP_EXTRACT and device.type == 'cuda'
    for batch in tqdm(loader, desc=f"extract {len(mods)} models"):
        xs = {s: batch[s].to(device, non_blocking=True) for s in sizes}
        B = next(iter(xs.values())).size(0)
        for n, m in mods.items():
            with torch.autocast('cuda', enabled=amp):
                f = m.encode(xs[Config.MODELS[n]['size']])
            out[n][pos:pos + B] = f.float().cpu().numpy()
        pos += B
    assert pos == N, (pos, N)
    os.makedirs(Config.FEATURE_DIR, exist_ok=True)
    for n in mods:
        np.save(feat_path(n), out[n])
    print(f"✓ extracted {list(mods)} in {(time.time()-t0)/60:.1f} min")
    del mods, out; gc.collect(); torch.cuda.empty_cache()


# ---------------------------------------------------------------- evaluation (same recipe as Part I)
def bootstrap_auroc_by_patient(y, p, pat, n_boot, seed):
    pats = np.unique(pat)
    idx_by = {q: np.where(pat == q)[0] for q in pats}
    rng, vals = np.random.RandomState(seed), []
    for _ in range(n_boot):
        s = rng.choice(pats, size=len(pats), replace=True)
        idx = np.concatenate([idx_by[q] for q in s])
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(roc_auc_score(y[idx], p[idx]))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)


def probe_auroc(X, y, centers, patients, train_c, test_c, with_ci=False):
    tr, te = np.isin(centers, train_c), centers == test_c
    sc = StandardScaler().fit(X[tr])
    clf = LogisticRegression(max_iter=Config.LR_MAX_ITER, C=Config.LR_C, class_weight='balanced',
                             random_state=Config.PROBE_SEED, n_jobs=-1)
    clf.fit(sc.transform(X[tr]), y[tr])
    prob = clf.predict_proba(sc.transform(X[te]))[:, 1]
    auc = roc_auc_score(y[te], prob)
    bal = balanced_accuracy_score(y[te], prob > 0.5)
    if with_ci:
        lo, hi = bootstrap_auroc_by_patient(y[te], prob, patients[te], Config.BOOT_N, Config.BOOT_SEED)
        return auc, bal, lo, hi
    return auc, bal


def center_id_probe(X, centers, patients):
    """Patch-level 5-way center-identity probe, patient-disjoint train/test within each center."""
    rng = np.random.RandomState(Config.PROBE_SEED)
    tr_idx, te_idx = [], []
    for c in Config.ALL_CENTERS:
        pats = sorted(np.unique(patients[centers == c]).tolist())
        random.Random(Config.PROBE_SEED + c).shuffle(pats)
        n_te = min(max(1, int(round(Config.CID_VAL_FRAC * len(pats)))), len(pats) - 1)
        te_p, tr_p = set(pats[:n_te]), set(pats[n_te:])
        for pool, n_want, store in ((np.where((centers == c) & np.isin(patients, list(tr_p)))[0], Config.CID_TRAIN_PER_CENTER, tr_idx),
                                    (np.where((centers == c) & np.isin(patients, list(te_p)))[0], Config.CID_TEST_PER_CENTER, te_idx)):
            store.append(rng.choice(pool, size=min(n_want, len(pool)), replace=False))
    tr_idx, te_idx = np.concatenate(tr_idx), np.concatenate(te_idx)
    sc = StandardScaler().fit(X[tr_idx])
    clf = LogisticRegression(max_iter=Config.LR_MAX_ITER, n_jobs=-1).fit(sc.transform(X[tr_idx]), centers[tr_idx])
    return 100 * accuracy_score(centers[te_idx], clf.predict(sc.transform(X[te_idx])))


def evaluate_model(name, y, centers, patients):
    cache = f"{Config.RESULTS_DIR}/{name}.json"
    if os.path.exists(cache):
        return json.load(open(cache))
    X = np.load(feat_path(name), mmap_mode='r')
    X = np.ascontiguousarray(X)
    t0 = time.time()
    auc, bal, lo, hi = probe_auroc(X, y, centers, patients, [c for c in Config.ALL_CENTERS if c != Config.TEST_CENTER],
                                   Config.TEST_CENTER, with_ci=True)
    loco = {}
    for ho in Config.LOCO_FOLDS:
        loco[ho] = probe_auroc(X, y, centers, patients, [c for c in Config.ALL_CENTERS if c != ho], ho)[0]
    vals = np.array(list(loco.values()))
    res = {'model': name, 'primary_auroc': auc, 'primary_ci_lo': lo, 'primary_ci_hi': hi, 'primary_balacc': bal,
           'loco_mean': float(vals.mean()), 'loco_std': float(vals.std(ddof=1)) if len(vals) > 1 else 0.0,
           'loco_folds': {str(k): float(v) for k, v in loco.items()},
           'center_id_acc': center_id_probe(X, centers, patients)}
    print(f"  [{name}] primary {auc:.4f} [{lo:.4f},{hi:.4f}] | LOCO {res['loco_mean']:.4f}±{res['loco_std']:.4f} | "
          f"center-ID {res['center_id_acc']:.1f}% | {(time.time()-t0)/60:.1f} min")
    os.makedirs(Config.RESULTS_DIR, exist_ok=True)
    json.dump(res, open(cache, 'w'))
    del X; gc.collect()
    return res


def _ms(v):
    v = np.array(v, dtype=float)
    return v.mean(), (v.std(ddof=1) if len(v) > 1 else float('nan'))


def summarize(results):
    """results: per-model dicts for the NEW seeds.  Seed 42 and ImageNet come from Config (first D run)."""
    per = {}                                   # (bb, mode) -> {seed: (primary, loco, cid)}
    for (bb, mode), v in Config.SEED42.items():
        per[(bb, mode)] = {42: v}
    for r in results:
        sp = Config.MODELS[r['model']]
        if sp['seed'] == 42:      # reproducibility check against the first D run
            old = Config.SEED42[(sp['backbone'], sp['mode'])]
            print(f"  seed-42 recheck {sp['backbone']:10s} {sp['mode']:9s} LOCO now {r['loco_mean']:.4f} vs first run {old[1]:.4f}")
        per[(sp['backbone'], sp['mode'])][sp['seed']] = (r['primary_auroc'], r['loco_mean'], r['center_id_acc'])
    rows = []
    for bb in ('resnet18', 'vit_small', 'swin_tiny'):
        for mode in ('neighbor', 'self'):
            d = per[(bb, mode)]
            pm, ps = _ms([d[s][0] for s in d]); lm, ls = _ms([d[s][1] for s in d]); cm, cs = _ms([d[s][2] for s in d])
            rows.append({'backbone': bb, 'method': mode, 'seeds': ','.join(str(s) for s in sorted(d)),
                         'LOCO mean': f"{lm:.4f} ± {ls:.4f}", 'primary AUROC': f"{pm:.4f} ± {ps:.4f}",
                         'center-ID %': f"{cm:.1f} ± {cs:.1f}"})
        im = Config.IMAGENET[bb]
        rows.append({'backbone': bb, 'method': 'ImageNet (1 run)', 'seeds': '-', 'LOCO mean': f"{im[1]:.4f}",
                     'primary AUROC': f"{im[0]:.4f}", 'center-ID %': f"{im[2]:.1f}"})
    df = pd.DataFrame(rows)
    print("\n" + "=" * 100 + "\nPHASE 2 -- MULTI-SEED SUMMARY (mean ± std across seeds; std uses ddof=1)\n" + "=" * 100)
    print(df.to_string(index=False))
    print("\nPer-seed LOCO mean:")
    for (bb, mode), d in per.items():
        print(f"  {bb:10s} {mode:9s} " + "  ".join(f"s{s}: {d[s][1]:.4f}" for s in sorted(d)))
    print("\nGain in LOCO mean AUROC vs same-architecture ImageNet (points, x100), mean ± std over seeds:")
    for bb in ('resnet18', 'vit_small', 'swin_tiny'):
        for mode in ('neighbor', 'self'):
            g = [100 * (v[1] - Config.IMAGENET[bb][1]) for v in per[(bb, mode)].values()]
            m, s = _ms(g)
            print(f"  {bb:10s} {mode:9s} {m:+.2f} ± {s:.2f}   (min {min(g):+.2f}, all seeds above ImageNet: {all(x > 0 for x in g)})")
    print("\nAdjacency minus self-control, paired by seed (LOCO x100):")
    for bb in ('resnet18', 'vit_small', 'swin_tiny'):
        dn, ds = per[(bb, 'neighbor')], per[(bb, 'self')]
        common = sorted(set(dn) & set(ds))
        g = [100 * (dn[s][1] - ds[s][1]) for s in common]
        m, sd = _ms(g)
        print(f"  {bb:10s} " + "  ".join(f"s{s}: {x:+.2f}" for s, x in zip(common, g)) +
              f"   mean {m:+.2f} ± {sd:.2f}   neighbor wins {sum(x > 0 for x in g)}/{len(g)}")
    os.makedirs(Config.RESULTS_DIR, exist_ok=True)
    df.to_csv(f"{Config.RESULTS_DIR}/phase2_multiseed_summary.csv", index=False)
    return df


def load_meta(hf):
    return (np.array(hf['label']).astype(np.int64), np.array(hf['center']).astype(np.int64),
            np.array(hf['patient']).astype(np.int64))


def main(hf=None, names=None):
    set_seed(Config.PROBE_SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"device: {device}")
    ensure_checkpoint_roots()
    if hf is None:
        from datasets import load_dataset, concatenate_datasets
        ds = load_dataset(Config.HF_DATASET)
        parts = [ds[k] for k in ds.keys()]
        hf = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    print(f"✓ {len(hf):,} patches")
    y, centers, patients = load_meta(hf)
    names = names or list(Config.MODELS)
    os.makedirs(Config.FEATURE_DIR, exist_ok=True); os.makedirs(Config.RESULTS_DIR, exist_ok=True)
    results = []
    for bb in ('resnet18', 'vit_small', 'swin_tiny'):          # one pass per architecture, saved when done
        group = [n for n in names if Config.MODELS[n]['backbone'] == bb]
        print(f"\n=== {bb}: {group}")
        extract_missing(hf, group, device)
        for n in group:
            if os.path.exists(feat_path(n)):
                results.append(evaluate_model(n, y, centers, patients))
            else:
                print(f"  [{n}] no features (checkpoint missing) -> not evaluated")
    return summarize(results)


# ====================================================================
# RUN
# ====================================================================
results_df = main()


device: cuda


README.md:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  470MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  469MB            

...

✓ extracted ['adjcl_resnet18_neighbor_s42', 'adjcl_resnet18_self_s42', 'adjcl_resnet18_neighbor_s43', 'adjcl_resnet18_self_s43', 'adjcl_resnet18_neighbor_s44', 'adjcl_resnet18_self_s44'] in 8.3 min
  [adjcl_resnet18_neighbor_s42] primary 0.9744 [0.9349,0.9864] | LOCO 0.9780±0.0066 | center-ID 44.2% | 5.2 min
  [adjcl_resnet18_self_s42] primary 0.9591 [0.9099,0.9739] | LOCO 0.9651±0.0066 | center-ID 42.6% | 6.0 min
  [adjcl_resnet18_neighbor_s43] primary 0.9667 [0.9077,0.9843] | LOCO 0.9784±0.0075 | center-ID 47.0% | 5.1 min
  [adjcl_resnet18_self_s43] primary 0.9578 [0.9069,0.9730] | LOCO 0.9664±0.0074 | center-ID 42.7% | 6.2 min
  [adjcl_resnet18_neighbor_s44] primary 0.9743 [0.9386,0.9857] | LOCO 0.9778±0.0063 | center-ID 46.9% | 4.7 min
  [adjcl_resnet18_self_s44] primary 0.9540 [0.8950,0.9716] | LOCO 0.9633±0.0079 | center-ID 42.4% | 6.2 min

=== vit_small: ['adjcl_vit_small_neighbor_s42', 'adjcl_vit_small_self_s42', 'adjcl_vit_small_neighbor_s43', 'adjcl_vit_small_self_s43', 'adjc

extract 6 models: 100%|██████████| 3563/3563 [2:25:23<00:00,  2.45s/it]


✓ extracted ['adjcl_vit_small_neighbor_s42', 'adjcl_vit_small_self_s42', 'adjcl_vit_small_neighbor_s43', 'adjcl_vit_small_self_s43', 'adjcl_vit_small_neighbor_s44', 'adjcl_vit_small_self_s44'] in 145.4 min
  [adjcl_vit_small_neighbor_s42] primary 0.9849 [0.9572,0.9929] | LOCO 0.9844±0.0075 | center-ID 78.1% | 3.2 min
  [adjcl_vit_small_self_s42] primary 0.9737 [0.9369,0.9848] | LOCO 0.9733±0.0053 | center-ID 70.5% | 3.8 min
  [adjcl_vit_small_neighbor_s43] primary 0.9735 [0.9073,0.9919] | LOCO 0.9793±0.0121 | center-ID 81.5% | 3.1 min
  [adjcl_vit_small_self_s43] primary 0.9731 [0.9339,0.9847] | LOCO 0.9705±0.0084 | center-ID 72.3% | 4.0 min
  [adjcl_vit_small_neighbor_s44] primary 0.9802 [0.9332,0.9937] | LOCO 0.9847±0.0051 | center-ID 82.2% | 3.1 min
  [adjcl_vit_small_self_s44] primary 0.9721 [0.9292,0.9852] | LOCO 0.9732±0.0062 | center-ID 71.0% | 3.8 min

=== swin_tiny: ['adjcl_swin_tiny_neighbor_s42', 'adjcl_swin_tiny_self_s42', 'adjcl_swin_tiny_neighbor_s43', 'adjcl_swin_tiny_se

extract 6 models: 100%|██████████| 3563/3563 [3:22:25<00:00,  3.41s/it]


✓ extracted ['adjcl_swin_tiny_neighbor_s42', 'adjcl_swin_tiny_self_s42', 'adjcl_swin_tiny_neighbor_s43', 'adjcl_swin_tiny_self_s43', 'adjcl_swin_tiny_neighbor_s44', 'adjcl_swin_tiny_self_s44'] in 202.6 min
  [adjcl_swin_tiny_neighbor_s42] primary 0.9879 [0.9756,0.9919] | LOCO 0.9834±0.0065 | center-ID 81.3% | 7.4 min
  [adjcl_swin_tiny_self_s42] primary 0.9771 [0.9491,0.9859] | LOCO 0.9748±0.0059 | center-ID 80.3% | 9.2 min
  [adjcl_swin_tiny_neighbor_s43] primary 0.9880 [0.9740,0.9923] | LOCO 0.9850±0.0058 | center-ID 81.8% | 6.9 min
  [adjcl_swin_tiny_self_s43] primary 0.9784 [0.9489,0.9873] | LOCO 0.9761±0.0060 | center-ID 78.8% | 9.4 min
  [adjcl_swin_tiny_neighbor_s44] primary 0.9875 [0.9742,0.9919] | LOCO 0.9834±0.0070 | center-ID 81.1% | 7.3 min
  [adjcl_swin_tiny_self_s44] primary 0.9753 [0.9471,0.9842] | LOCO 0.9747±0.0052 | center-ID 78.3% | 8.9 min
  seed-42 recheck resnet18   neighbor  LOCO now 0.9780 vs first run 0.9780
  seed-42 recheck resnet18   self      LOCO now 0.965